# Knee MRI: 10-model submission (5 baseline folds + 5 v4mix folds), with timing
Inputs to attach: competition data, `knee-mri-pipeline` dataset (v5 code), the baseline training output (fold models),
the v4mix notebook output (`models_v4mix/`), the caching notebook output (`labels_soft.csv`, OLD labels, fallback priors only).
GPU on, internet OFF. Every number printed under TIMING is measured, not guessed.
On the visible commit run the test set is tiny, so cell 5 benchmarks 100 TRAIN studies and projects the full-scoring time.
That benchmark is skipped automatically when the test set has 50+ studies (the real scoring run), so it never inflates the runtime.

In [1]:
import os, re, sys, time, json, shutil, subprocess
from pathlib import Path
import numpy as np, pandas as pd
T0 = time.time()
ROOT = Path(os.environ.get("KAGGLE_INPUT_ROOT", "/kaggle/input"))
W = Path(os.environ.get("KNEE_WORK", "/kaggle/working")); W.mkdir(parents=True, exist_ok=True)

# ---- overrides: leave None to auto-discover; set them if a cell below stops with "ambiguous" ----
CODE_OVERRIDE      = None   # folder containing infer.py, model.py, cache_data.py
BASE_DIRS_OVERRIDE = None   # list of folders with the BASELINE plain fold0..4.pt (e.g. modelg0, modelg1)
V4_DIR_OVERRIDE    = None   # folder with fold0_v4mix.pt .. fold4_v4mix.pt
LABELS_OVERRIDE    = None   # OLD labels_soft.csv (only used for fallback priors of unreadable studies)
N_TEST_FULL_EST    = 1300   # assumed size of the hidden test set, only for the projection

SKIP = {"train_series", "test_series", "cache", "__pycache__", ".git"}
def walk(root, maxdepth=6):
    q = [(Path(root), 0)]
    while q:
        d, k = q.pop(0)
        try: ents = sorted(os.scandir(d), key=lambda e: e.name)
        except OSError: continue
        for e in ents:
            if e.is_dir(follow_symlinks=False):
                if e.name not in SKIP and k < maxdepth: q.append((Path(e.path), k + 1))
            else: yield Path(e.path)

files = list(walk(ROOT))
print("input files scanned:", len(files))

def pick(cands, what, override):
    if override: return Path(override)
    cands = sorted(set(cands))
    print(f"{what}: {len(cands)} found", *[f"\n   {c}" for c in cands])
    assert len(cands) == 1, f"ambiguous or missing {what}: set the *_OVERRIDE variable above"
    return cands[0]

# code
code_dirs = [f.parent for f in files if f.name == "infer.py" and (f.parent / "model.py").exists() and (f.parent / "cache_data.py").exists()]
CODE = pick(code_dirs, "code folder (infer.py)", CODE_OVERRIDE)
assert "KneeNet" in (CODE / "model.py").read_text(), "model.py has no KneeNet"

# competition data
data_dirs = [f.parent for f in files if f.name == "test_series.csv" and (f.parent / "sample_submission.csv").exists() and (f.parent / "test.csv").exists()]
DATA = pick(data_dirs, "competition data folder", None)

# fold models
v4 = {}; base = {}
for f in files:
    m = re.fullmatch(r"fold(\d)_v4mix\.pt", f.name)
    if m: v4.setdefault(f.parent, {})[int(m.group(1))] = f
    m = re.fullmatch(r"fold(\d)\.pt", f.name)
    if m and "v4mix" not in str(f).lower(): base.setdefault(f.parent, {})[int(m.group(1))] = f
print("\nv4mix model folders:", {str(k): sorted(v) for k, v in v4.items()})
print("baseline model folders:", {str(k): sorted(v) for k, v in base.items()})
v4_dir = pick(list(v4), "v4mix model folder", V4_DIR_OVERRIDE)
V4 = {i: f for i, f in (v4[v4_dir] if not V4_DIR_OVERRIDE else
      {int(re.search(r"fold(\d)", p.name).group(1)): p for p in Path(V4_DIR_OVERRIDE).glob("fold*_v4mix.pt")}).items()}
if BASE_DIRS_OVERRIDE:
    BASE = {}
    for d in BASE_DIRS_OVERRIDE:
        for p in Path(d).glob("fold?.pt"): BASE[int(p.stem[-1])] = p
else:
    BASE = {}
    for d, fm in base.items():
        for i, f in fm.items():
            assert i not in BASE, f"baseline fold {i} appears in two folders ({BASE[i].parent} and {d}): set BASE_DIRS_OVERRIDE"
            BASE[i] = f
assert sorted(V4) == [0, 1, 2, 3, 4], f"v4mix folds found: {sorted(V4)}"
assert sorted(BASE) == [0, 1, 2, 3, 4], f"baseline folds found: {sorted(BASE)} (need exactly 0..4, set BASE_DIRS_OVERRIDE)"
print("\nbaseline:", *[f"\n   fold{i}: {BASE[i]}" for i in range(5)])
print("v4mix:   ", *[f"\n   fold{i}: {V4[i]}" for i in range(5)])

# labels for fallback priors (old labels only)
lab = [f for f in files if f.name == "labels_soft.csv"]
LABELS = Path(LABELS_OVERRIDE) if LABELS_OVERRIDE else None
if LABELS is None:
    ok = []
    for f in lab:
        try:
            h = pd.read_csv(f, nrows=3)
            if "y_ACL" in h.columns: ok.append(f)
        except Exception: pass
    print("labels_soft.csv candidates:", ok)
    LABELS = ok[0] if len(ok) == 1 else None
print("fallback-prior labels:", LABELS, "(None -> 0.5 for unreadable studies)")
print(f"\nSETUP OK  [{time.time()-T0:.0f}s]")

input files scanned: 68
code folder (infer.py): 1 found 
   /kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline
competition data folder: 1 found 
   /kaggle/input/competitions/rsna-knee-abnormality-detection

v4mix model folders: {'/kaggle/input/notebooks/theokorir/knee-v4-mix-5fold/models_v4mix': [0, 1, 2, 3, 4]}
baseline model folders: {'/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-training/models_g0': [0, 1, 2], '/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-training/models_g1': [3, 4]}
v4mix model folder: 1 found 
   /kaggle/input/notebooks/theokorir/knee-v4-mix-5fold/models_v4mix

baseline: 
   fold0: /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-training/models_g0/fold0.pt 
   fold1: /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-training/models_g0/fold1.pt 
   fold2: /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-training/models_g0/fold2.pt 
   fold3: /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-training/models_g1/fold3.pt 


In [2]:
# ---- assemble ONE folder of 10 checkpoints with unique names and verify them ----
import torch
ALL = W / "models_all"; shutil.rmtree(ALL, ignore_errors=True); ALL.mkdir()
rows = []
for tag, src in (("base", BASE), ("v4mix", V4)):
    for i in range(5):
        dst = ALL / f"fold{i}_{tag}.pt"; shutil.copy(src[i], dst)
        ck = torch.load(dst, map_location="cpu")
        rows.append(dict(file=dst.name, backbone=ck["backbone"], K=ck["K"], n_tensors=len(ck["state"]), MB=round(dst.stat().st_size/1e6, 1)))
M = pd.DataFrame(rows); print(M.to_string(index=False))
assert len(M) == 10 and M.backbone.nunique() == 1 and M.K.nunique() == 1, "checkpoints disagree on backbone/K: infer.py would silently use the last K"
print(f"\n10 checkpoints OK | backbone {M.backbone[0]} | K {M.K[0]}")

          file        backbone  K  n_tensors   MB
 fold0_base.pt efficientnet_b0  1        402 23.1
 fold1_base.pt efficientnet_b0  1        402 23.1
 fold2_base.pt efficientnet_b0  1        402 23.1
 fold3_base.pt efficientnet_b0  1        402 23.1
 fold4_base.pt efficientnet_b0  1        402 23.1
fold0_v4mix.pt efficientnet_b0  1        402 23.1
fold1_v4mix.pt efficientnet_b0  1        402 23.1
fold2_v4mix.pt efficientnet_b0  1        402 23.1
fold3_v4mix.pt efficientnet_b0  1        402 23.1
fold4_v4mix.pt efficientnet_b0  1        402 23.1

10 checkpoints OK | backbone efficientnet_b0 | K 1


In [3]:
# ---- run inference (timed) ----
test_ids = pd.read_csv(DATA / "test.csv").StudyInstanceUID
N_TEST = len(test_ids)
print("test studies in this run:", N_TEST)

def run_infer(data_dir, out, tag):
    cmd = [sys.executable, str(CODE / "infer.py"), "--models_dir", str(ALL), "--data_dir", str(data_dir),
           "--out", str(out), "--workers", "4", "--bs", "4"]
    if LABELS: cmd += ["--labels", str(LABELS)]
    t = time.time(); last = ""
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=str(CODE))
    for line in p.stdout:
        last = line.rstrip()
        if "/" in last and "|" in last and last.split("/")[0].isdigit():
            n = int(last.split("/")[0])
            if n % 40 == 0: print(f"[{tag}] {last}", flush=True)
        else: print(f"[{tag}] {last}", flush=True)
    rc = p.wait(); wall = time.time() - t
    assert rc == 0, f"infer.py failed (exit {rc})"
    m = re.search(r"\| (\d+)s\s*$", last)
    loop = float(m.group(1)) if m else float("nan")      # infer.py's own time for the data+model loop
    return wall, loop

t_main = time.time()
wall_main, loop_main = run_infer(DATA, W / "submission.csv", "main")
print(f"\nmain run: wall {wall_main:.0f}s | inference loop {loop_main:.0f}s | load/setup ~{wall_main-loop_main:.0f}s | per study {loop_main/max(N_TEST,1):.2f}s")

test studies in this run: 3
[main] /kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline/model.py:49: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
[main]   self.fuse = nn.TransformerEncoder(layer, n_layers)
[main] saved /kaggle/working/submission.csv (3, 13) | unreadable studies: 0 | 10s

main run: wall 31s | inference loop 10s | load/setup ~21s | per study 3.33s


In [4]:
# ---- benchmark for the projection: ONLY when the test set is tiny (commit run). Skipped in the real scoring run. ----
BENCH_N = 100
bench = None
if N_TEST < 50:
    tr = pd.read_csv(DATA / "train.csv"); ts = pd.read_csv(DATA / "train_series.csv")
    ids = tr.StudyInstanceUID.sample(BENCH_N, random_state=0).tolist()
    B = Path("/tmp/bench"); shutil.rmtree(B, ignore_errors=True); B.mkdir(parents=True)
    pd.DataFrame({"StudyInstanceUID": ids}).to_csv(B / "test.csv", index=False)
    ts[ts.StudyInstanceUID.isin(ids)].to_csv(B / "test_series.csv", index=False)
    sub0 = pd.read_csv(DATA / "sample_submission.csv"); cols = list(sub0.columns)
    sb = pd.DataFrame(0.5, index=range(BENCH_N), columns=cols); sb["StudyInstanceUID"] = ids; sb.to_csv(B / "sample_submission.csv", index=False)
    os.symlink(DATA / "train_series", B / "test_series")
    wall_b, loop_b = run_infer(B, B / "bench_sub.csv", "bench")
    per = loop_b / BENCH_N; load_s = wall_b - loop_b
    proj = load_s + per * N_TEST_FULL_EST
    bench = dict(per_study=per, load=load_s, proj=proj)
    print(f"\n===== TIMING PROJECTION (10 models, THIS GPU/CPU) =====")
    print(f"  model load + setup : {load_s:6.0f} s")
    print(f"  per study          : {per:6.2f} s   (100 train studies)")
    print(f"  projected for {N_TEST_FULL_EST} studies: {proj:7.0f} s = {proj/3600:.2f} h   (limit 9 h = 32400 s)")
    print(f"  efficiency runtime term RuntimeSeconds/32400 = {proj/32400:.3f}")
    print(f"  rule of thumb: +0.01 AUC is worth ~900-1000 s of runtime on the Efficiency Track")
    print("  caveat: scoring hardware may differ from this run; train studies may decode a bit differently from test.")
else:
    print(f"test set has {N_TEST} studies: real scoring run, benchmark skipped.")

[bench] /kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline/model.py:49: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
[bench]   self.fuse = nn.TransformerEncoder(layer, n_layers)
[bench] 40/100 | 35s
[bench] 80/100 | 53s
[bench] saved /tmp/bench/bench_sub.csv (100, 13) | unreadable studies: 0 | 68s

===== TIMING PROJECTION (10 models, THIS GPU/CPU) =====
  model load + setup :     15 s
  per study          :   0.68 s   (100 train studies)
  projected for 1300 studies:     899 s = 0.25 h   (limit 9 h = 32400 s)
  efficiency runtime term RuntimeSeconds/32400 = 0.028
  rule of thumb: +0.01 AUC is worth ~900-1000 s of runtime on the Efficiency Track
  caveat: scoring hardware may differ from this run; train studies may decode a bit differently from test.


In [5]:
# ---- final validation of submission.csv ----
sub = pd.read_csv(W / "submission.csv"); ref = pd.read_csv(DATA / "sample_submission.csv")
assert list(sub.columns) == list(ref.columns), "columns differ from sample_submission"
assert set(sub.StudyInstanceUID) == set(ref.StudyInstanceUID) and len(sub) == len(ref), "study ids differ"
vals = sub.drop(columns="StudyInstanceUID")
assert vals.notna().all().all() and ((vals >= 0) & (vals <= 1)).all().all(), "NaN or out-of-range values"
print(f"({len(sub)}, {sub.shape[1]}) OK")
try:   # cosmetic only: must never be able to fail the notebook
    print(f"min over labels of prediction std: {float(vals.std().min()):.3f} (should not be ~0)")
except Exception as e:
    print("std check skipped:", repr(e))
print(f"TOTAL notebook time so far: {time.time()-T0:.0f}s")

(3, 13) OK
min over labels of prediction std: 0.033 (should not be ~0)
TOTAL notebook time so far: 124s
